# 03 - Gold Aggregations

Creates business-facing Gold Delta datasets from the Silver layer.

In [ ]:
%run ../utilities/config

In [ ]:
from pyspark.sql.functions import col, count, countDistinct, sum, to_date

silver_df = spark.read.format("delta").load(SILVER_PATH)

In [ ]:
# Daily revenue

gold_daily_revenue = (
    silver_df
    .filter(col("event_type") == "payment_success")
    .withColumn("event_date", to_date(col("event_timestamp")))
    .groupBy("event_date")
    .agg(
        sum("price").alias("total_revenue"),
        count("*").alias("successful_payments"),
    )
)

gold_daily_revenue.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/daily_revenue")
display(gold_daily_revenue)

In [ ]:
# User activity

gold_user_activity = (
    silver_df
    .withColumn("event_date", to_date(col("event_timestamp")))
    .groupBy("event_date")
    .agg(
        countDistinct("user_id").alias("active_users"),
        count("*").alias("total_events"),
    )
)

gold_user_activity.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/user_activity")
display(gold_user_activity)

In [ ]:
# Event funnel

gold_event_funnel = (
    silver_df
    .groupBy("event_type")
    .agg(count("*").alias("event_count"))
    .orderBy(col("event_count").desc())
)

gold_event_funnel.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/event_funnel")
display(gold_event_funnel)

In [ ]:
# Failed payments

gold_failed_payments = (
    silver_df
    .filter(col("event_type") == "payment_failed")
    .withColumn("event_date", to_date(col("event_timestamp")))
    .groupBy("event_date")
    .agg(
        count("*").alias("failed_payment_count"),
        sum("price").alias("failed_payment_value"),
    )
)

gold_failed_payments.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/failed_payments")
display(gold_failed_payments)

In [ ]:
# Verify Gold outputs

display(spark.read.format("delta").load(f"{GOLD_PATH}/daily_revenue"))
display(spark.read.format("delta").load(f"{GOLD_PATH}/user_activity"))
display(spark.read.format("delta").load(f"{GOLD_PATH}/event_funnel"))
display(spark.read.format("delta").load(f"{GOLD_PATH}/failed_payments"))